# 卷积神经网络实践：训练与测试

使用完整的 PyTorch 代码训练 Fashion-MNIST 服饰分类模型。
按顺序运行各单元即可完成环境检查、数据读取、模型训练、参数保存和最终测试。
模型、训练循环与测试代码均已提供。

环境安装见项目根目录 `README.md`，操作说明见 `任务书_卷积神经网络实践课.md`。
CPU 可运行完整流程，兼容设备可切换为 CUDA 或 MPS。


## 1. 导入依赖

Jupyter 内核选择 `Python (cv-cnn)`。打印的解释器路径应属于 `cv-cnn` 环境。


In [ ]:
from pathlib import Path
import os, sys, json, re
import numpy as np
import torch
import torchvision
from torch import nn
import matplotlib.pyplot as plt

LAB_ROOT = next((p for p in (Path.cwd(), Path.cwd().parent)
                 if (p / 'cnn_common.py').is_file()), None)
if LAB_ROOT is None:
    raise FileNotFoundError('请从实践课_卷积神经网络目录启动 Jupyter。')
sys.path.insert(0, str(LAB_ROOT))
from cnn_common import (CLASS_NAMES, set_seed, select_device, make_loaders,
                        make_test_loader, evaluate, fit, count_parameters,
                        show_batch, plot_history, plot_confusion, show_mistakes)
print('Python:', sys.version.split()[0], '| interpreter:', sys.executable)
print('torch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available(), '| runtime:', torch.version.cuda)
print('MPS available:', torch.backends.mps.is_available())


## 2. 设置运行配置

默认使用 CPU，训练 10,000 张、验证 2,000 张，共 3 轮。
初次检查流程可改成 256/128 张和 1 轮，并将 `RUN_NAME` 改为 `smoke`。
CPU 计算较慢时可使用 5,000/1,000 张；增加轮数时可将 `EPOCHS` 改为 5 或 10。
不同运行使用不同名称可保留各自结果，同名输出会被覆盖。


In [ ]:
DEVICE_CHOICE = 'cpu'            # 可选 'cpu' / 'cuda' / 'mps' / 'auto'
TRAIN_SIZE, VAL_SIZE = 10000, 2000
EPOCHS, BATCH_SIZE = 3, 64
LEARNING_RATE = 1e-3
SEED = 2026
RUN_NAME = 'baseline'
DOWNLOAD = True                  # 完整数据已缓存后可改 False
NUM_THREADS = min(4, os.cpu_count() or 1)
DATA_ROOT = LAB_ROOT / 'data'
OUTPUT_ROOT = LAB_ROOT / 'outputs'
if min(TRAIN_SIZE, VAL_SIZE, EPOCHS, BATCH_SIZE) < 1:
    raise ValueError('样本数、训练轮数和 batch size 必须大于 0。')
if not re.fullmatch(r'[A-Za-z0-9_-]+', RUN_NAME):
    raise ValueError('RUN_NAME 只能包含英文字母、数字、下划线或连字符。')
if not 0 < LEARNING_RATE < float('inf'):
    raise ValueError('学习率必须是大于 0 的有限数值。')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
checkpoint_path = OUTPUT_ROOT / f'{RUN_NAME}_best.pt'
torch.set_num_threads(NUM_THREADS)
device = select_device(DEVICE_CHOICE)
set_seed(SEED)
probe = torch.ones(2, 2, device=device, requires_grad=True)
probe.square().sum().backward()
print('实际运行设备:', device, '| 张量设备:', probe.device)
print('张量运算与自动求导结果:', probe.grad)


## 3. 读取与查看数据

图像为单通道 `28×28`，共 10 类。训练与验证数据从官方训练集划分，索引互不重叠。
`ToTensor()` 将像素转成 `[0,1]` 的浮点数，`Normalize((0.5,), (0.5,))` 再映射到 `[-1,1]`。
测试数据不参与训练和模型选择。

首次运行会下载并解压约 30 MB 的压缩数据；也可在终端运行 `python prepare_data.py` 准备完整缓存。


In [ ]:
train_loader, val_loader = make_loaders(
    DATA_ROOT, TRAIN_SIZE, VAL_SIZE, BATCH_SIZE, SEED, DOWNLOAD, device)
train_ids = set(train_loader.dataset.indices)
val_ids = set(val_loader.dataset.indices)
assert train_ids.isdisjoint(val_ids)
images, labels = next(iter(train_loader))
print('train / val:', len(train_loader.dataset), len(val_loader.dataset))
print('训练/验证索引交集:', len(train_ids & val_ids))
print('images:', tuple(images.shape), images.dtype,
      '| range:', images.min().item(), images.max().item())
print('labels:', tuple(labels.shape), labels.dtype)
show_batch(train_loader)


## 4. 完整模型与训练循环

两组卷积与池化将图像变成 `32×7×7` 的特征，再通过全连接层输出 10 类 logits。
网络末尾不加 Softmax，直接使用 `CrossEntropyLoss`。
下方提供完整 `SimpleCNN` 和 `train_one_epoch`，与 `cnn_model.py` 的源码一致。

每个 batch 按“清梯度 → 前向 → 损失 → 反向 → 参数更新”的顺序执行；
图像、标签与模型都放在同一设备上。


In [ ]:
"""完整的 CNN 模型和单轮训练函数，可由 Notebook 或命令行直接调用。"""
import torch
from torch import nn


class SimpleCNN(nn.Module):
    """将 [N,1,28,28] 的灰度图像映射为 [N,10] 的分类 logits。"""

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, images):
        # CrossEntropyLoss 直接接收 logits，模型末尾无需 Softmax。
        return self.classifier(self.features(images))


def train_one_epoch(model, loader, criterion, optimizer, device):
    """完成一轮参数更新，返回按实际样本数加权的平均损失和准确率。"""
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    if total == 0:
        raise ValueError("训练数据为空。")
    return {"loss": total_loss / total, "acc": total_correct / total}


In [ ]:
set_seed(SEED)
preview_model = SimpleCNN().to(device)
with torch.no_grad():
    sample_logits = preview_model(images[:4].to(device))
print(preview_model)
print('输入形状:', tuple(images[:4].shape))
print('输出形状:', tuple(sample_logits.shape))
print('可训练参数量:', count_parameters(preview_model))
assert sample_logits.shape == (min(4, images.size(0)), 10)


## 5. 运行训练并保存最佳模型

训练使用训练集更新参数，每轮结束后在验证集上评估。
公共 `fit` 函数会保存验证准确率最高轮次的参数，结束后恢复该组参数。
验证使用 `model.eval()` 和 `torch.no_grad()`，不会更新网络参数。
验证准确率相同时保留较早轮次。


In [ ]:
# 重新固定种子和创建加载器，使前面的预览不改变正式训练的初始化与批顺序。
set_seed(SEED)
train_loader, val_loader = make_loaders(
    DATA_ROOT, TRAIN_SIZE, VAL_SIZE, BATCH_SIZE, SEED, DOWNLOAD, device)
model = SimpleCNN().to(device)
config = {
    'run_name': RUN_NAME, 'model': 'SimpleCNN', 'seed': SEED,
    'train_size': TRAIN_SIZE, 'val_size': VAL_SIZE,
    'batch_size': BATCH_SIZE, 'epochs': EPOCHS, 'lr': LEARNING_RATE,
    'device_requested': DEVICE_CHOICE, 'device': str(device),
    'data_root': str(DATA_ROOT), 'output_dir': str(OUTPUT_ROOT),
    'num_threads': NUM_THREADS, 'python_version': sys.version.split()[0],
    'torch_version': str(torch.__version__),
    'torchvision_version': str(torchvision.__version__),
    'parameters': count_parameters(model),
}
(OUTPUT_ROOT / f'{RUN_NAME}_config.json').write_text(
    json.dumps(config, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
history = fit(model, train_loader, val_loader, train_one_epoch, device,
              epochs=EPOCHS, lr=LEARNING_RATE, checkpoint=checkpoint_path, seed=SEED)
(OUTPUT_ROOT / f'{RUN_NAME}_history.json').write_text(
    json.dumps(history, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
plot_history(history, save_path=OUTPUT_ROOT / f'{RUN_NAME}_curves.png')
best = max(history, key=lambda row: row['val_acc'])
print(f"最佳验证准确率: {best['val_acc']:.2%} | 轮次: {best['epoch']}")
print('已保存模型:', checkpoint_path)


## 6. 加载模型并完成测试

配置确定后，加载最佳参数，在官方独立测试集上评估。
测试不更新模型参数。模型可加载到与训练不同的设备，`map_location` 会进行映射。
此处重新建立网络并加载参数，用于确认保存的模型可以独立使用。


In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
final_model = SimpleCNN().to(device)
final_model.load_state_dict(checkpoint['model_state'])
test_loader = make_test_loader(DATA_ROOT, BATCH_SIZE, DOWNLOAD, device)
result = evaluate(final_model, test_loader, nn.CrossEntropyLoss(), device,
                  return_predictions=True)
test_metrics = {
    'checkpoint': str(checkpoint_path), 'device': str(device),
    'test_samples': len(test_loader.dataset),
    'test_loss': result['loss'], 'test_accuracy': result['acc'],
    'best_epoch': checkpoint['epoch'], 'best_val_accuracy': checkpoint['val_acc'],
    'train_size': TRAIN_SIZE, 'val_size': VAL_SIZE,
    'parameters': count_parameters(final_model),
    'class_names': CLASS_NAMES, 'torch_version': str(torch.__version__),
}
(OUTPUT_ROOT / f'{RUN_NAME}_test_metrics.json').write_text(
    json.dumps(test_metrics, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('最佳轮次:', checkpoint['epoch'], '| 测试样本数:', len(test_loader.dataset))
print(f"测试损失: {result['loss']:.4f} | 测试准确率: {result['acc']:.2%}")
confusion = plot_confusion(result['true'], result['pred'],
                          save_path=OUTPUT_ROOT / f'{RUN_NAME}_confusion.png')
show_mistakes(final_model, test_loader, device,
              save_path=OUTPUT_ROOT / f'{RUN_NAME}_mistakes.png')


## 7. 查看运行结果

默认输出包括：

| 文件 | 内容 |
|---|---|
| `outputs/baseline_best.pt` | 最佳模型参数 |
| `outputs/baseline_config.json` | 运行配置 |
| `outputs/baseline_history.json` | 每轮训练与验证指标 |
| `outputs/baseline_curves.png` | 损失和准确率曲线 |
| `outputs/baseline_test_metrics.json` | 测试指标 |
| `outputs/baseline_confusion.png` | 混淆矩阵 |
| `outputs/baseline_mistakes.png` | 误分类图像 |

`RUN_NAME` 改变后，输出文件的前缀随之改变。
检查损失是否总体下降、验证准确率是否提高，以及保存的模型能否重新加载并测试。
增加轮数或调整学习率时，依据验证集表现判断效果。

| 项目 | 实际结果 |
|---|---|
| 运行设备 | |
| 训练/验证/测试样本数 | |
| 训练轮数、batch size、学习率 | |
| 最佳验证准确率与轮次 | |
| 测试准确率 | |
| 模型文件路径 | |

保留 Notebook 的运行输出、训练曲线、模型参数和配置记录，即可复查本次实验。
